In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("UberDataset Current.csv")


In [2]:
df["START_DATE"] = pd.to_datetime(df["START_DATE"])
df["date"] = df["START_DATE"].dt.date

In [3]:
demand_df = (
    df.groupby(["date", "hour"])
      .agg(
          ride_demand=("START_DATE", "size"),
          day_of_week=("day_of_week", "first"),
          is_weekend=("is_weekend", "first"),
          is_rush_hour=("is_rush_hour", "first"),
          temperature=("TEMPERATURE", "mean"),
          rain=("RAIN", "mean"),
          snowfall=("SNOWFALL", "mean"),
          showers=("SHOWERS", "mean"),
          is_day=("IS_DAY", "first"),
          weather_code=("WEATHER_CODE", "first"),
          wind_speed=("WIND_SPEED_10M", "mean"),
          wind_gusts=("WIND_GUSTS_10M", "mean"),
          temp_wind=("TEMP_WIND", "mean"),
          weather_stress=("WEATHER_STRESS", "mean")
      )
      .reset_index()
)

demand_df.head()

,date,hour,ride_demand,day_of_week,is_weekend,is_rush_hour,temperature,rain,snowfall,showers,is_day,weather_code,wind_speed,wind_gusts,temp_wind,weather_stress
0,2016-01-01,21,1,4,False,False,80.0,0.0,0.0,0.0,0.0,1.0,4.1,6.8,328.00,91.9
1,2016-01-02,1,1,5,False,False,75.5,0.0,0.0,0.0,0.0,1.0,9.2,13.3,694.60,99.0
2,2016-01-02,20,1,5,False,False,71.8,0.0,0.0,0.0,0.0,2.0,13.7,25.9,983.66,113.4
3,2016-01-05,17,1,1,False,True,61.3,0.0,0.0,0.0,1.0,3.0,21.4,42.1,1311.82,127.8
4,2016-01-06,14,1,2,False,False,67.4,0.7,0.0,0.0,1.0,53.0,18.8,37.8,1267.12,177.7


In [4]:
print("Original rides:", len(df))
print("Hourly observations:", len(demand_df))

print("\nRide demand distribution:")
print(demand_df["ride_demand"].value_counts().sort_index())

print("\nSummary:")
print(demand_df["ride_demand"].describe())

Original rides: 1154
Hourly observations: 1012

Ride demand distribution:
ride_demand
1    878
2    126
3      8
Name: count, dtype: int64

Summary:
count    1012.000000
mean        1.140316
std         0.369558
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: ride_demand, dtype: float64


In [5]:
print("Earliest ride:", df["START_DATE"].min())
print("Latest ride:", df["START_DATE"].max())

print("\nNumber of unique dates:", df["date"].nunique())

date_counts = (
    df.groupby("date")
      .size()
)

print("\nRides per active day:")
print(date_counts.describe())

print("\nFirst 10 days:")
print(date_counts.head(10))

Earliest ride: 2016-01-01 21:11:00
Latest ride: 2016-12-31 22:08:00

Number of unique dates: 294

Rides per active day:
count    294.00000
mean       3.92517
std        2.32977
min        1.00000
25%        2.00000
50%        4.00000
75%        5.00000
max       13.00000
dtype: float64

First 10 days:
date
2016-01-01    1
2016-01-02    2
2016-01-05    1
2016-01-06    3
2016-01-07    1
2016-01-10    5
2016-01-11    4
2016-01-12    6
2016-01-13    2
2016-01-14    2
dtype: int64


In [6]:
# Count rides for each date
daily_rides = (
    df.groupby("date")
      .size()
      .rename("ride_demand")
)

# Create EVERY date in 2016, including days with 0 rides
all_dates = pd.date_range(
    start=df["START_DATE"].min().normalize(),
    end=df["START_DATE"].max().normalize(),
    freq="D"
)

daily_df = pd.DataFrame({
    "date": all_dates
})

daily_df["date"] = daily_df["date"].dt.date

# Add ride counts
daily_df = daily_df.merge(
    daily_rides,
    left_on="date",
    right_index=True,
    how="left"
)

# Days that didn't appear in the original dataset had 0 rides
daily_df["ride_demand"] = (
    daily_df["ride_demand"]
    .fillna(0)
    .astype(int)
)

# Create temporal features
daily_df["date"] = pd.to_datetime(daily_df["date"])

daily_df["day_of_week"] = daily_df["date"].dt.dayofweek
daily_df["is_weekend"] = daily_df["day_of_week"].isin([5, 6])

daily_df.head()

,date,ride_demand,day_of_week,is_weekend
0,2016-01-01,1,4,False
1,2016-01-02,2,5,True
2,2016-01-03,0,6,True
3,2016-01-04,0,0,False
4,2016-01-05,1,1,False


In [7]:
print("Daily observations:", len(daily_df))

print("\nDaily demand distribution:")
print(
    daily_df["ride_demand"]
    .value_counts()
    .sort_index()
)

print("\nDaily demand summary:")
print(
    daily_df["ride_demand"]
    .describe()
)

print(
    "\nZero-ride days:",
    (daily_df["ride_demand"] == 0).sum()
)

Daily observations: 366

Daily demand distribution:
ride_demand
0     72
1     35
2     62
3     49
4     45
5     45
6     21
7     10
8     12
9      5
10     6
11     3
13     1
Name: count, dtype: int64

Daily demand summary:
count    366.000000
mean       3.153005
std        2.607382
min        0.000000
25%        1.000000
50%        3.000000
75%        5.000000
max       13.000000
Name: ride_demand, dtype: float64

Zero-ride days: 72


In [8]:
# Additional temporal features
daily_df["month"] = daily_df["date"].dt.month
daily_df["day_of_month"] = daily_df["date"].dt.day

# Optional cyclical representations
daily_df["day_of_week_sin"] = np.sin(
    2 * np.pi * daily_df["day_of_week"] / 7
)

daily_df["day_of_week_cos"] = np.cos(
    2 * np.pi * daily_df["day_of_week"] / 7
)

daily_df.head()

,date,ride_demand,day_of_week,is_weekend,month,day_of_month,day_of_week_sin,day_of_week_cos
0,2016-01-01,1,4,False,1,1,-0.433884,-0.900969
1,2016-01-02,2,5,True,1,2,-0.974928,-0.222521
2,2016-01-03,0,6,True,1,3,-0.781831,0.623490
3,2016-01-04,0,0,False,1,4,0.000000,1.000000
4,2016-01-05,1,1,False,1,5,0.781831,0.623490


In [9]:
split_index = int(len(daily_df) * 0.8)

train_df = daily_df.iloc[:split_index].copy()
test_df = daily_df.iloc[split_index:].copy()

print("Training observations:", len(train_df))
print("Testing observations:", len(test_df))

print(
    "Training period:",
    train_df["date"].min(),
    "to",
    train_df["date"].max()
)

print(
    "Testing period:",
    test_df["date"].min(),
    "to",
    test_df["date"].max()
)

Training observations: 292
Testing observations: 74
Training period: 2016-01-01 00:00:00 to 2016-10-18 00:00:00
Testing period: 2016-10-19 00:00:00 to 2016-12-31 00:00:00


In [10]:
from sklearn.metrics import mean_absolute_error

# Average training demand for each day of week
dow_average = (
    train_df
    .groupby("day_of_week")["ride_demand"]
    .mean()
)

print("Historical average demand by day:")
print(dow_average)

Historical average demand by day:
day_of_week
0    3.166667
1    3.238095
2    2.219512
3    2.487805
4    3.952381
5    2.404762
6    2.452381
Name: ride_demand, dtype: float64


In [11]:
test_df["baseline_prediction"] = (
    test_df["day_of_week"]
    .map(dow_average)
)

baseline_mae = mean_absolute_error(
    test_df["ride_demand"],
    test_df["baseline_prediction"]
)

print(f"Historical baseline MAE: {baseline_mae:.3f}")

Historical baseline MAE: 2.349


In [12]:
from sklearn.ensemble import RandomForestRegressor

temporal_features = [
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "month",
    "day_of_month"
]

X_train = train_df[temporal_features]
X_test = test_df[temporal_features]

y_train = train_df["ride_demand"]
y_test = test_df["ride_demand"]

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=3
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

model_mae = mean_absolute_error(
    y_test,
    predictions
)

print(f"Historical baseline MAE: {baseline_mae:.3f}")
print(f"Random Forest MAE:       {model_mae:.3f}")
mae_improvement = (
    (baseline_mae - model_mae)
    / baseline_mae
) * 100

print(
    f"MAE improvement over baseline: "
    f"{mae_improvement:.1f}%"
)

Historical baseline MAE: 2.349
Random Forest MAE:       2.352
MAE improvement over baseline: -0.1%


In [13]:
weather_daily = (
    df.groupby("date")
      .agg(
          temperature=("TEMPERATURE", "mean"),
          rain=("RAIN", "mean"),
          snowfall=("SNOWFALL", "mean"),
          showers=("SHOWERS", "mean"),
          wind_speed=("WIND_SPEED_10M", "mean"),
          wind_gusts=("WIND_GUSTS_10M", "mean"),
          temp_wind=("TEMP_WIND", "mean"),
          weather_stress=("WEATHER_STRESS", "mean")
      )
      .reset_index()
)

weather_daily["date"] = pd.to_datetime(
    weather_daily["date"]
)

daily_df = daily_df.merge(
    weather_daily,
    on="date",
    how="left"
)

daily_df[
    [
        "date",
        "ride_demand",
        "temperature",
        "rain",
        "snowfall"
    ]
].head(10)

,date,ride_demand,temperature,rain,snowfall
0,2016-01-01,1,80.000000,0.000000,0.0
1,2016-01-02,2,73.650000,0.000000,0.0
2,2016-01-03,0,NaN,NaN,NaN
3,2016-01-04,0,NaN,NaN,NaN
4,2016-01-05,1,61.300000,0.000000,0.0
5,2016-01-06,3,69.733333,0.566667,0.0
6,2016-01-07,1,48.200000,0.000000,0.0
7,2016-01-08,0,NaN,NaN,NaN
8,2016-01-09,0,NaN,NaN,NaN
9,2016-01-10,5,45.040000,0.000000,0.0


In [14]:
weather_features = [
    "temperature",
    "rain",
    "snowfall",
    "showers",
    "wind_speed",
    "wind_gusts",
    "temp_wind",
    "weather_stress"
]

print(daily_df[weather_features].isna().sum())

temperature       91
rain              91
snowfall          91
showers           91
wind_speed        91
wind_gusts        91
temp_wind         91
weather_stress    72
dtype: int64


In [15]:
# Days where rides occurred but weather data is missing
missing_weather_days = daily_df[
    daily_df["temperature"].isna()
]

print("Total days missing temperature:", len(missing_weather_days))

print(
    "Missing-weather days with 0 rides:",
    (missing_weather_days["ride_demand"] == 0).sum()
)

print(
    "Missing-weather days with >0 rides:",
    (missing_weather_days["ride_demand"] > 0).sum()
)

print("\nDays with rides but missing weather:")
print(
    missing_weather_days.loc[
        missing_weather_days["ride_demand"] > 0,
        ["date", "ride_demand"]
    ]
)
rides_missing_weather = df[
    df["TEMPERATURE"].isna()
]

print("Rides missing temperature:", len(rides_missing_weather))

print(
    rides_missing_weather[
        [
            "START_DATE",
            "MILES",
            "TEMPERATURE",
            "RAIN",
            "SNOWFALL",
            "WIND_SPEED_10M"
        ]
    ].head(30)
)

Total days missing temperature: 91
Missing-weather days with 0 rides: 72
Missing-weather days with >0 rides: 19

Days with rides but missing weather:
          date  ride_demand
230 2016-08-18            1
236 2016-08-24            2
245 2016-09-02            2
248 2016-09-05            1
249 2016-09-06            1
253 2016-09-10            1
254 2016-09-11            2
255 2016-09-12            3
256 2016-09-13            1
257 2016-09-14            1
258 2016-09-15            1
259 2016-09-16            1
261 2016-09-18            1
272 2016-09-29            1
277 2016-10-04            2
282 2016-10-09            1
284 2016-10-11            1
351 2016-12-17            2
352 2016-12-18            3
Rides missing temperature: 162
             START_DATE  MILES  TEMPERATURE  RAIN  SNOWFALL  WIND_SPEED_10M
294 2016-02-16 08:29:00   14.1          NaN   NaN       NaN             NaN
295 2016-02-17 13:18:00   14.7          NaN   NaN       NaN             NaN
296 2016-08-05 18:17:00    1.8 

In [18]:
import pandas as pd
import json
import glob
from pathlib import Path

weather_folder = "Week 2/*.json"

json_files = glob.glob(weather_folder)

print("JSON files found:", len(json_files))

weather_lookup = {}

for file in json_files:
    location = Path(file).stem

    with open(file, "r") as f:
        data = json.load(f)

    hourly = data["hourly"]

    location_weather = pd.DataFrame({
        "datetime": pd.to_datetime(hourly["time"]),
        "TEMPERATURE": hourly["apparent_temperature"],
        "IS_DAY": hourly["is_day"],
        "RAIN": hourly["rain"],
        "SNOWFALL": hourly["snowfall"],
        "SHOWERS": hourly["showers"],
        "WEATHER_CODE": hourly["weather_code"],
        "WIND_SPEED_10M": hourly["wind_speed_10m"],
        "WIND_GUSTS_10M": hourly["wind_gusts_10m"]
    })

    # Makes date/hour lookup fast
    location_weather["datetime"] = (
        location_weather["datetime"].dt.floor("h")
    )

    location_weather = location_weather.set_index("datetime")

    weather_lookup[location] = location_weather

ride_locations = set(df["START"].dropna().unique())
json_locations = set(weather_lookup.keys())

missing_json_locations = ride_locations - json_locations

print("Ride locations:", len(ride_locations))
print("JSON locations:", len(json_locations))

print("\nRide locations without JSON files:")
print(sorted(missing_json_locations))

JSON files found: 169
Ride locations: 174
JSON locations: 169

Ride locations without JSON files:
['Kalorama Triangle', 'NO_DATA', 'Noorpur Shahan', 'SOMISSPO', 'Savon Height']


In [21]:
missing_locations = [
    "Kalorama Triangle",
    "NO_DATA",
    "Noorpur Shahan",
    "SOMISSPO",
    "Savon Height"
]

print(
    df[df["START"].isin(missing_locations)]
    ["START"]
    .value_counts()
)

print("\nTotal rides:")
print(df["START"].isin(missing_locations).sum())

missing_weather_rides = df[df["TEMPERATURE"].isna()]

print(
    missing_weather_rides[
        missing_weather_rides["START"].isin(missing_locations)
    ][["START_DATE", "START", "STOP", "MILES"]]
)

print("\nCounts:")
print(
    missing_weather_rides[
        missing_weather_rides["START"].isin(missing_locations)
    ]["START"].value_counts()
)

START
NO_DATA              148
Noorpur Shahan         5
Savon Height           4
Kalorama Triangle      3
SOMISSPO               2
Name: count, dtype: int64

Total rides:
162
              START_DATE              START              STOP  MILES
294  2016-02-16 08:29:00            NO_DATA           Colombo   14.1
295  2016-02-17 13:18:00            NO_DATA           Colombo   14.7
296  2016-08-05 18:17:00  Kalorama Triangle  Columbia Heights    1.8
314  2016-08-03 16:00:00  Kalorama Triangle          Downtown    1.5
387  2016-10-19 13:45:00           SOMISSPO    French Quarter    1.7
...                  ...                ...               ...    ...
1041 2016-12-23 14:15:00            NO_DATA           NO_DATA    9.6
1042 2016-12-23 16:23:00            NO_DATA           NO_DATA    1.3
1043 2016-12-31 15:03:00            NO_DATA           NO_DATA   16.2
1117 2016-05-23 21:09:00       Savon Height       Whitebridge    3.6
1118 2016-10-31 21:45:00       Savon Height       Whitebridge    9

In [23]:
# Pick the first ride that is missing weather
test_row = df[df["TEMPERATURE"].isna()].iloc[0]

print("START_DATE:", test_row["START_DATE"])
print("START location:", repr(test_row["START"]))
print("Location exists in lookup:", test_row["START"] in weather_lookup)

location = test_row["START"]

if location in weather_lookup:
    ride_hour = test_row["START_DATE"].floor("h")

    print("Ride hour:", ride_hour)
    print("Ride hour type:", type(ride_hour))

    location_weather = weather_lookup[location]

    print("\nWeather index type:")
    print(type(location_weather.index[0]))

    print("\nFirst weather timestamps:")
    print(location_weather.index[:5])

    print("\nRide hour exists in weather index:")
    print(ride_hour in location_weather.index)

START_DATE: 2016-02-16 08:29:00
START location: 'NO_DATA'
Location exists in lookup: False


In [25]:
import os
import json
import pandas as pd

# Make sure row labels match 0, 1, 2, ...
df = df.reset_index(drop=True)

df["START_DATE"] = pd.to_datetime(df["START_DATE"])

cache_dir = "cached"

weather_columns = [
    "TEMPERATURE",
    "IS_DAY",
    "RAIN",
    "SNOWFALL",
    "SHOWERS",
    "WEATHER_CODE",
    "WIND_SPEED_10M",
    "WIND_GUSTS_10M"
]

filled = 0
no_json = []
no_timestamp = []

for idx, row in df.iterrows():

    # Only fix rows that are currently missing weather
    if pd.notna(row["TEMPERATURE"]):
        continue

    location = row["START"]

    cache_file = os.path.join(
        cache_dir,
        f"{location}.json"
    )

    # JSON doesn't exist for this location
    if not os.path.exists(cache_file):
        no_json.append((idx, location))
        continue

    with open(cache_file, "r") as f:
        full_json = json.load(f)

    hourly = full_json["hourly"]

    # EXACTLY like the Week 2 notebook
    target_str = row["START_DATE"].strftime(
        "%Y-%m-%dT%H:00"
    )

    # Make sure that timestamp exists
    if target_str not in hourly["time"]:
        no_timestamp.append(
            (idx, location, target_str)
        )
        continue

    find_idx = hourly["time"].index(target_str)

    df.at[idx, "TEMPERATURE"] = (
        hourly["apparent_temperature"][find_idx]
    )

    df.at[idx, "IS_DAY"] = (
        hourly["is_day"][find_idx]
    )

    df.at[idx, "RAIN"] = (
        hourly["rain"][find_idx]
    )

    df.at[idx, "SNOWFALL"] = (
        hourly["snowfall"][find_idx]
    )

    df.at[idx, "SHOWERS"] = (
        hourly["showers"][find_idx]
    )

    df.at[idx, "WEATHER_CODE"] = (
        hourly["weather_code"][find_idx]
    )

    df.at[idx, "WIND_SPEED_10M"] = (
        hourly["wind_speed_10m"][find_idx]
    )

    df.at[idx, "WIND_GUSTS_10M"] = (
        hourly["wind_gusts_10m"][find_idx]
    )

    filled += 1

print("Rows successfully filled:", filled)

print("\nRemaining missing weather:")
print(df[weather_columns].isna().sum())

print("\nLocations without JSON:")
print(
    pd.Series(
        [location for _, location in no_json]
    ).value_counts()
)

print("\nTimestamp lookup failures:")
print(len(no_timestamp))

Rows successfully filled: 0

Remaining missing weather:
TEMPERATURE       162
IS_DAY            162
RAIN              162
SNOWFALL          162
SHOWERS           162
WEATHER_CODE      162
WIND_SPEED_10M    162
WIND_GUSTS_10M    162
dtype: int64

Locations without JSON:
NO_DATA              148
Noorpur Shahan         5
Savon Height           4
Kalorama Triangle      3
SOMISSPO               2
Name: count, dtype: int64

Timestamp lookup failures:
0


In [26]:
missing_df = df[df["TEMPERATURE"].isna()].copy()

missing_df[
    ["START_DATE", "START", "STOP"]
].head(30)

missing_df["stop_has_json"] = (
    missing_df["STOP"].isin(weather_lookup.keys())
)

print(
    missing_df["stop_has_json"]
    .value_counts()
)

print(
    "\nMissing rides recoverable using STOP:",
    missing_df["stop_has_json"].sum()
)

print(
    "Still unrecoverable:",
    (~missing_df["stop_has_json"]).sum()
)

filled_from_stop = 0

for idx, row in df.iterrows():

    # Already has weather
    if pd.notna(row["TEMPERATURE"]):
        continue

    stop_location = row["STOP"]

    if stop_location not in weather_lookup:
        continue

    target_str = row["START_DATE"].strftime(
        "%Y-%m-%dT%H:00"
    )

    location_weather = weather_lookup[stop_location]

    # Convert datetime index back to matching Timestamp
    target_time = row["START_DATE"].floor("h")

    if target_time not in location_weather.index:
        continue

    weather_row = location_weather.loc[target_time]

    df.at[idx, "TEMPERATURE"] = weather_row["TEMPERATURE"]
    df.at[idx, "IS_DAY"] = weather_row["IS_DAY"]
    df.at[idx, "RAIN"] = weather_row["RAIN"]
    df.at[idx, "SNOWFALL"] = weather_row["SNOWFALL"]
    df.at[idx, "SHOWERS"] = weather_row["SHOWERS"]
    df.at[idx, "WEATHER_CODE"] = weather_row["WEATHER_CODE"]
    df.at[idx, "WIND_SPEED_10M"] = weather_row["WIND_SPEED_10M"]
    df.at[idx, "WIND_GUSTS_10M"] = weather_row["WIND_GUSTS_10M"]

    filled_from_stop += 1

print("Filled using STOP location:", filled_from_stop)

print(
    df[weather_columns]
    .isna()
    .sum()
)

remaining = df[df["TEMPERATURE"].isna()]

print("\nRemaining missing rides:", len(remaining))

print("\nRemaining START locations:")
print(remaining["START"].value_counts())

print("\nRemaining STOP locations:")
print(remaining["STOP"].value_counts())

stop_has_json
False    93
True     69
Name: count, dtype: int64

Missing rides recoverable using STOP: 69
Still unrecoverable: 93
Filled using STOP location: 69
TEMPERATURE       93
IS_DAY            93
RAIN              93
SNOWFALL          93
SHOWERS           93
WEATHER_CODE      93
WIND_SPEED_10M    93
WIND_GUSTS_10M    93
dtype: int64

Remaining missing rides: 93

Remaining START locations:
START
NO_DATA           90
Noorpur Shahan     2
SOMISSPO           1
Name: count, dtype: int64

Remaining STOP locations:
STOP
NO_DATA           88
Noorpur Shahan     4
French Quarter     1
Name: count, dtype: int64


In [28]:
daily_df["rides_previous_day"] = (
    daily_df["ride_demand"].shift(1)
)

daily_df["rides_previous_week"] = (
    daily_df["ride_demand"].shift(7)
)

daily_df["rolling_7d_avg"] = (
    daily_df["ride_demand"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_df["rolling_14d_avg"] = (
    daily_df["ride_demand"]
    .shift(1)
    .rolling(14)
    .mean()
)

lag_df = daily_df.dropna(
    subset=[
        "rides_previous_day",
        "rides_previous_week",
        "rolling_7d_avg",
        "rolling_14d_avg"
    ]
).copy()

print("Observations after lag creation:", len(lag_df))
split_index = int(len(lag_df) * 0.8)

train_lag = lag_df.iloc[:split_index].copy()
test_lag = lag_df.iloc[split_index:].copy()

print("Training observations:", len(train_lag))
print("Testing observations:", len(test_lag))

lag_features = [
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "month",
    "day_of_month",
    "rides_previous_day",
    "rides_previous_week",
    "rolling_7d_avg",
    "rolling_14d_avg"
]

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train_lag = train_lag[lag_features]
X_test_lag = test_lag[lag_features]

y_train_lag = train_lag["ride_demand"]
y_test_lag = test_lag["ride_demand"]

lag_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=3
)

lag_model.fit(
    X_train_lag,
    y_train_lag
)

lag_predictions = lag_model.predict(
    X_test_lag
)

lag_mae = mean_absolute_error(
    y_test_lag,
    lag_predictions
)

print(f"Lag model MAE: {lag_mae:.3f}")

dow_average_lag = (
    train_lag
    .groupby("day_of_week")["ride_demand"]
    .mean()
)

baseline_lag_predictions = (
    test_lag["day_of_week"]
    .map(dow_average_lag)
)

baseline_lag_mae = mean_absolute_error(
    y_test_lag,
    baseline_lag_predictions
)

print(f"Baseline MAE:  {baseline_lag_mae:.3f}")
print(f"Lag model MAE: {lag_mae:.3f}")


lag_improvement = (
    (baseline_lag_mae - lag_mae)
    / baseline_lag_mae
) * 100

print(
    f"MAE improvement over baseline: "
    f"{lag_improvement:.1f}%"
)

Observations after lag creation: 352
Training observations: 281
Testing observations: 71
Lag model MAE: 2.119
Baseline MAE:  2.310
Lag model MAE: 2.119
MAE improvement over baseline: 8.3%
